# Simulationsgestützte Optimierung mit SciPy

Optimierung mithilfe von scipy.optimize


* battery_kWh =  300
* pv_kWp = 192



In [6]:


from Model import *
from scipy.optimize import minimize, differential_evolution
import time


trucks = [Etruck("workday_lunchbreak") for i in range(7)]
    #trucks+= [Etruck("worknight") for i in range(2)]
    #trucks+= [t1 for i in range(10)]
    #trucks+= [t2 for i in range(10)]

results = simulate(start_day=0, 
                   hours=8760, 
                   trucks=trucks, 
                   battery_kWh=300, 
                   pv_kWp=192,
                   monthly_km=[68000,61000,66000,
                                   63000,65000,64000,
                                   64000,62000,60000,
                                   68000,65000,68000,],
                   grid_threshold=0.99,
                   )
print(results)
print(f'Total cost: {results.system_cost + results.operating_cost*10:.0f} €/10a')

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
Energy Flows: 
Grid                527905.0
PV to Truck          61938.0
PV to Battery        66663.0
PV to Grid           70762.0
Battery to Truck     66647.0
Grid to Truck       527905.0
Driven             -774000.0
dtype: float64
PV Yield: 199363 kWh/a
____________________
Total cost: 2_967_566 €/10a
self.system_cost=378_000.0€
self.operating_cost=258_956.6€/a
self.emissions=142534.3kg/a
____________________
Self-consumption: 65%
Load Cycles: 222.2
Total cost: 2967566 €/10a


Definieren Sie eine Funktion *objective_function_PV(x)*, welche als Argument *x* die Leistung (kWp) der PV-Anlage erhält, den Wert entsprechend im Simulationsmodell setzt, die Simulation ausführt und die berechneten Gesamtkosten (system costs + operating costs über 10 Jahre) als Ergebnis zurückliefert. Der Parameter battery_kWh = 300 soll dabei unverändert bleiben. (10 %)

In [7]:
def objective_function_PV(x):
    results = simulate(start_day=0, 
                   hours=8760, 
                   trucks=trucks, 
                   battery_kWh=300, 
                   pv_kWp=x,
                   monthly_km=[68000,61000,66000,
                               63000,65000,64000,
                               64000,62000,60000,
                               68000,65000,68000,],
                   grid_threshold=0.99,
                   )
    return (results.system_cost + results.operating_cost*10)

print(objective_function_PV(1))

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
3375808.6833437504


## Aufgabe 2

Programmieren Sie eine einfache Grid Search (in 1D), indem Sie in einer for-Schleife die oben definierte Funktion *objective_function_PV()* für verschiedene kWp-Werte (im Bereich 50-800 kW in 50 kW-Schritten) aufrufen und die Rückgabewerte ausgeben (Funktion *print*). Bei welchem dieser kWp-Werte sind die berechneten Gesamtkosten am geringsten? (10 %)

In [8]:
for i in range(5,801,50):
    r = objective_function_PV(i)
    print(f"{i} kWp, Cost: {r}")

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
5 kWp, Cost: 3361043.4167187503
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
55 kWp, Cost: 3197550.9804047504
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
105 kWp, Cost: 3086674.3395681004
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
155 kWp, Cost: 3013944.096734512
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2

## Aufgabe 3

Verwenden Sie die Funktion *scipy.optimize.minimize()* aus dem Python-Package *scipy* um den optimalen kWp-Wert für die PV-Anlage zu finden, der die Gesamtkosten minimiert. Verwenden Sie die oben definierte Funktion *objective_function_PV()* als Zielfunktion und legen Sie geeignete Grenzwerte (*bounds*) sowie einen sinnvollen Startwert *x0* fest.

(Hinweis: Unter Umständen kann der Rechenvorgang ein paar Minuten dauern.)

Falls Sie mit der Default-Methode Probleme haben, testen Sie auch andere Optimierungsverfahren (speziell 'trust-constr' oder auch 'CG'), indem Sie die Funktion *minimize* mit der ensprechenden Option *method='trust-constr'* bzw. method='CG' aufrufen. Ggf. kann es auch helfen die Toleranzen enger zu setzen (z.B. Option *tol=1e-6*). Nähere Details dazu können Sie in der Hilfe nachlesen.

Lassen Sie sich den Rückgabewert von *minimize()* in der Konsole ausgeben. Wie lautet der berechnete optimale kWp-Wert? (15 %)

In [9]:
bounds = [(200,500)]
#options={'gtol': 1e-16, 'eps': 1e-20, 'ftol': 2e-16}

res = minimize(
    objective_function_PV,
    x0 = 300,
    bounds = bounds,
    #method = "Nelder-Mead",
    #method = 'CG',
    #method = 'L-BFGS-B',
    method = 'trust-constr',
    #options = options
    #tol=1e-6
    )
print("done.")
res

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


           message: `gtol` termination condition is satisfied.
           success: True
            status: 1
               fun: 2842492.9022604614
                 x: [ 5.000e+02]
               nit: 9
              nfev: 16
              njev: 8
              nhev: 0
          cg_niter: 7
      cg_stop_cond: 1
              grad: [-1.012e+02]
   lagrangian_grad: [ 1.577e-12]
            constr: [array([ 5.000e+02])]
               jac: [<1x1 sparse matrix of type '<class 'numpy.float64'>'
                    	with 1 stored elements in Compressed Sparse Row format>]
       constr_nfev: [0]
       constr_njev: [0]
       constr_nhev: [0]
                 v: [array([ 1.012e+02])]
            method: tr_interior_point
        optimality: 1.5774048733874224e-12
  constr_violation: 0.0
    execution_time: 2.7018532752990723
         tr_radius: 4954.438221222418
    constr_penalty: 1.0
 barrier_parameter: 0.020000000000000004
 barrier_tolerance: 0.020000000000000004
             niter: 9

In [10]:
res4 = differential_evolution(
    objective_function_PV,
    bounds=bounds,
    popsize=20,
    mutation=0.05,
    recombination=0.05)
print("done.")
res4

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


 message: Optimization terminated successfully.
 success: True
     fun: 2842492.8822604967
       x: [ 5.000e+02]
     nit: 1
    nfev: 44
     jac: [-1.012e+02]

## Aufgabe 4

Definieren Sie eine zweite Funktion *objective_function_2D(x)*, welche als Inputwert x einen Vektor (Liste) aus den beiden Parametern
* Dimenstionierung (kWp) der PV-Anlage
* Batteriekapazität

erhält, die Parameter entsprechend im Simulationsmodell setzt, die Simulation ausführt und die berechneten Gesamtkosten (system costs + operating costs über 10 Jahre) als Ergebnis zurückliefert. (10 %)

In [12]:
def objective_function_2D(x):
    results = simulate(start_day=0, 
                   hours=8760, 
                   trucks=trucks, 
                   battery_kWh=x[1], 
                   pv_kWp=x[0],
                   monthly_km=[68000,61000,66000,
                               63000,65000,64000,
                               64000,62000,60000,
                               68000,65000,68000,],
                   grid_threshold=0.99,
                   )
    return (results.system_cost + results.operating_cost*10)

print(objective_function_2D((5,10)))

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
3274784.682635095


## Aufgabe 5

Verwenden Sie wieder die Funktion *scipy.optimize.minimize()* aus dem Package scipy um sowohl den kWp-Wert für die PV-Anlage als auch die Batteriekapzazität gleichzeitig zu optimieren. Verwenden Sie dazu die oben definierte Funktion *objective_function_2D()* als Zielfunktion und legen Sie geeignete Grenzwerte (*bounds*) sowie sinnvolle Startwerte *x0* fest.
Testen Sie auch andere Optimierungsverfahren (speziell 'trust-constr') und vergleichen Sie die Ergebnisse. (15 %)

In [13]:
bounds2 = [(200,800), (400,900)]
#options={'gtol': 1e-6, 'eps': 1e-12, 'ftol': 2e-9}

res2 = minimize(
    objective_function_2D,
    x0 =(400,700),
    bounds = bounds2,
    #method = "Nelder-Mead",
    #method = 'CG',
    method = 'trust-constr'
    #options = options
    )
print("done.")
res2

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


c:\ProgramData\anaconda3\Lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


c:\ProgramData\anaconda3\Lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


...

c:\ProgramData\anaconda3\Lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


c:\ProgramData\anaconda3\Lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


           message: `xtol` termination condition is satisfied.
           success: True
            status: 2
               fun: 2670524.126161478
                 x: [ 6.362e+02  9.000e+02]
               nit: 251
              nfev: 1026
              njev: 342
              nhev: 0
          cg_niter: 264
      cg_stop_cond: 2
              grad: [ 2.344e+00 -1.346e+01]
   lagrangian_grad: [ 2.344e+00 -4.767e-04]
            constr: [array([ 6.362e+02,  9.000e+02])]
               jac: [<2x2 sparse matrix of type '<class 'numpy.float64'>'
                    	with 2 stored elements in Compressed Sparse Row format>]
       constr_nfev: [0]
       constr_njev: [0]
       constr_nhev: [0]
                 v: [array([-9.967e-05,  1.346e+01])]
            method: tr_interior_point
        optimality: 2.344004770657264
  constr_violation: 0.0
    execution_time: 215.72742009162903
         tr_radius: 9.765625000000004e-09
    constr_penalty: 1.0
 barrier_parameter: 2.048000000000001e-09


## Aufgabe 6

Verwenden Sie anstelle von *minimize()* die Funktion *scipy.optimize.differential_evolution()* um das Optimum mittels Differential-Evolution-Verfahren zu berechnen. Legen Sie dazu wieder geeignete Grenzen (*bounds*) fest. Vergleichen Sie die Ergebnisse mit jenen aus Aufgabe 5.
> Hinweis: Wie genau die Funktion differential_evolution aufgerufen werden kann (und welche Input-Argumente sie erlaubt bzw. benötigt), können Sie in der Dokumentation nachlesen: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.differential_evolution.html

Probieren Sie auch verschiedene Verfahrensparameter aus, speziell die Populationsgröße (*popsize*) sowie Mutations- und Rekombinationsraten (*mutation*, *recombination*). (15 %)

In [15]:
res3 = differential_evolution(
    objective_function_2D,
    bounds=[(200,1000), (200,1000)],
    popsize=20,
    mutation=0.05,
    recombination=0.05,
    tol=1e-3
    )
print("done.")
res3

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


 message: Optimization terminated successfully.
 success: True
     fun: 2669446.2436602036
       x: [ 6.379e+02  9.580e+02]
     nit: 8
    nfev: 438

## Aufgabe 7

Vergleichen Sie die Rechenzeit sowie die Anzahl der Funktionsauswertungen 
* zwischen *minimize()* im 1D-Fall (Aufgabe 3) und im 2D-Fall (Aufgabe 5),
* zwischen *minimize()* (Aufgabe 5) und *differential_evolution()* (Aufgabe 6), 

indem die Werte jeweils in einer Tabelle gegenüberstellen. Welches der Verfahren ist im jeweiligen Fall effizienter? (10 %)
> Hinweis: Die Anzahl der Funktionsauswertungen (*nfev*) wird Ihnen von den beiden Funktionen jeweils mit zurück geliefert. Um die Rechenzeit zu bestimmen, vergleichen Sie die Zeiten vor und nach dem Funktionsaufruf. Verwenden Sie dazu das Python-Package *time*:  
>> *import time  
>> t = time.time()  
>> (...Funktionsaufruf...)  
>> elapsed_time = time.time() - t*  



In [16]:
import time

# 1D
t = time.time()
res = minimize(objective_function_PV, x0 = 300, bounds = bounds, method = 'trust-constr')
elapsed_time_1D = time.time() - t
print("done.")

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


In [17]:
# 2D
t = time.time()
res2 = minimize(objective_function_2D, x0 = (300,300), bounds = bounds2, method = 'trust-constr')
elapsed_time_2D = time.time() - t
print("done.")

[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


c:\ProgramData\anaconda3\Lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


c:\ProgramData\anaconda3\Lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


...

c:\ProgramData\anaconda3\Lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


c:\ProgramData\anaconda3\Lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]
[2193.548387096774, 2178.5714285714284, 2129.032258064516, 2100.0, 2096.7741935483873, 2133.3333333333335, 2064.516129032258, 2000.0, 2000.0, 2193.548387096774, 2166.6666666666665, 2193.548387096774]


KeyboardInterrupt: 

In [ ]:
# DE
t = time.time()
res3 = differential_evolution(objective_function_2D, bounds=bounds2, popsize=20, mutation=0.05, recombination=0.05)
elapsed_time_DE = time.time() - t
print("done.")

In [ ]:
from tabulate import tabulate
headers = [' ', 'runtime', 'nfev']
tab1 = [['1D', elapsed_time_1D, res.nfev],['2D', elapsed_time_2D, res2.nfev]]
tab2 = [['2D', elapsed_time_2D, res2.nfev],['DE', elapsed_time_DE, res3.nfev]]

print(tabulate(tab1, headers=headers))
print(' ')
print(tabulate(tab2, headers=headers))

## Aufgabe 8

Variieren Sie andere Parameterwerte aus der Simulation, speziell
* Energiepreise,
* Trucks Anzahl und schedules,

in sinnvollen Bereichen per Hand und führen Sie die Optimierungsläufe von Aufgabe 5 nochmals durch. Wie stark ändern sich die optimalen PV- und Batteriewerte?  
(15 %)